# Bright and dark modes from SLURM MCWF

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "common").is_dir() and (candidate / "slurm").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

analysis_dir = repo_root / "slurm" / "analyses" / "bright_dark_modes_mcwf"
input_dir = analysis_dir / "output"

from common.plotting import (
    plot_bloch_angles,
    plot_harmonic_sweep,
    plot_spin_components,
)
from parser.mcwf import TrajectoryEnsemble
from parser.mfe import MFESolverParameters
from parser.moments import MomentSeries, SimulationMetadata
from post_analysis import (
    compute_harmonic_analysis,
    compute_j_modes_from_j_moments,
    compute_j_moments,
    compute_mean_generator_moments,
    extract_family_phase_series,
)
from slurm.trajectory_ensemble_io import load_trajectory_ensemble_artifact
from solvers.mfe import compute_mfe_j_moments, solve_mfe

## Input artifact

In [ ]:
# Match the job name in submit.slurm; the single run uses artifact index zero.
run_name = "bright-dark-v1"
artifact_path = input_dir / f"{run_name}_0.pkl"
save_plots = False
output_dir = input_dir / "plots"

if not artifact_path.is_file():
    raise FileNotFoundError(f"Missing simulation artifact: {artifact_path}")
if save_plots:
    output_dir.mkdir(parents=True, exist_ok=True)

artifact_path

## MCWF moments and equivalent MFE solution

In [ ]:
artifact = load_trajectory_ensemble_artifact(artifact_path)
required_fields = {"ensemble", "simulation_metadata", "array_index", "ntraj"}
missing_fields = required_fields.difference(artifact)
if missing_fields:
    raise ValueError(
        f"{artifact_path} is missing fields: {sorted(missing_fields)}"
    )

ensemble = artifact["ensemble"]
metadata = artifact["simulation_metadata"]
if not isinstance(ensemble, TrajectoryEnsemble):
    raise TypeError(f"{artifact_path} does not contain a TrajectoryEnsemble.")
if not isinstance(metadata, SimulationMetadata):
    raise TypeError(f"{artifact_path} does not contain SimulationMetadata.")
if artifact["array_index"] != 0:
    raise ValueError(f"{artifact_path} has the wrong single-run index.")
if artifact["ntraj"] != ensemble.moments.num_trajectories:
    raise ValueError(f"{artifact_path} has inconsistent trajectory counts.")
if ensemble.metadata is None:
    raise ValueError(f"{artifact_path} has no ensemble metadata.")
if len(metadata.Ni) != 2:
    raise ValueError("This analysis requires two physical groups.")
if len(metadata.phase_protocol.family_phases) != 3:
    raise ValueError("This analysis requires the three-phase protocol.")

raw_moments = ensemble.moments
ntraj = raw_moments.num_trajectories
sectors = tuple(ensemble.metadata.sectors)
mcwf_moments = MomentSeries(t=raw_moments.t.copy(), metadata=metadata)
mcwf_moments.mean_generator_moments = compute_mean_generator_moments(
    raw_moments,
)
mcwf_moments.J = compute_j_moments(
    mcwf_moments.mean_generator_moments,
    metadata=mcwf_moments.metadata,
)
# Release the full trajectory arrays after their covariance has been computed.
del artifact, ensemble, raw_moments

# MFE uses the saved MCWF physical inputs, protocol, and time grid.
mfe_moments = MomentSeries(t=mcwf_moments.t.copy(), metadata=metadata)
mfe_parameters = MFESolverParameters(
    Ni=metadata.Ni,
    omega_i=metadata.omega_groups,
    Gamma=metadata.Gamma,
    phase_protocol=metadata.phase_protocol,
)
mfe_result = solve_mfe(mfe_parameters, t_eval=mfe_moments.t)
if not mfe_result.success:
    raise RuntimeError(mfe_result.message)
mfe_moments.J = compute_mfe_j_moments(mfe_result)

for moments in (mcwf_moments, mfe_moments):
    moments.J_modes = compute_j_modes_from_j_moments(
        moments.J,
        omega_groups=moments.metadata.omega_groups,
    )

N_total = sum(metadata.Ni)
delta0 = metadata.phase_protocol.family_phases[1].delta
delta_factor = delta0 / (N_total * metadata.Gamma)
delta_label = rf"$\delta/(N\Gamma)={delta_factor:g}$"
method_series = (
    ("MCWF", mcwf_moments, "-"),
    ("MFE", mfe_moments, "--"),
)
print(f"Ni={metadata.Ni}, couplings={metadata.omega_groups}")
print(f"{ntraj} trajectories; {len(sectors)} initial sectors")

## Aggregated J angles and moments

In [ ]:
aggregated_angle_axes = None
aggregated_component_axes = None
for method_name, moments, linestyle in method_series:
    J = moments.J.aggregated
    aggregated_angle_fig, aggregated_angle_axes = plot_bloch_angles(
        moments.t,
        J.theta,
        J.phi,
        labels=(method_name,),
        colour_family_index=0,
        shade_index=2,
        linestyle=linestyle,
        axes=aggregated_angle_axes,
        phase_protocol=metadata.phase_protocol,
        title="Aggregated J-sphere angles",
    )
    aggregated_component_fig, aggregated_component_axes = plot_spin_components(
        moments.t,
        J.x,
        J.y,
        J.z,
        J.length,
        x_sem=J.x_sem,
        y_sem=J.y_sem,
        z_sem=J.z_sem,
        length_sem=J.length_sem,
        labels=(method_name,),
        colour_family_index=0,
        shade_index=2,
        linestyle=linestyle,
        axes=aggregated_component_axes,
        phase_protocol=metadata.phase_protocol,
        title="Aggregated J-sphere moments",
    )

# Keep time labels clear of the shared figure legends.
aggregated_angle_fig.supxlabel("")
aggregated_angle_axes[-1].set_xlabel(r"$\Gamma t$")
aggregated_component_fig.supxlabel("")
for ax in aggregated_component_axes[2:]:
    ax.set_xlabel(r"$\Gamma t$")

if save_plots:
    aggregated_angle_fig.savefig(
        output_dir / "aggregated_j_angles.png", dpi=200, bbox_inches="tight"
    )
    aggregated_component_fig.savefig(
        output_dir / "aggregated_j_moments.png", dpi=200, bbox_inches="tight"
    )

## Group-resolved J angles and moments

In [ ]:
group_angle_axes = None
group_component_axes = None
for method_name, moments, linestyle in method_series:
    J = moments.J
    labels = tuple(
        f"{method_name} group {group + 1}" for group in range(len(metadata.Ni))
    )
    group_angle_fig, group_angle_axes = plot_bloch_angles(
        J.t,
        J.theta,
        J.phi,
        labels=labels,
        shade_index=2,
        linestyle=linestyle,
        axes=group_angle_axes,
        phase_protocol=metadata.phase_protocol,
        title="Group-resolved J-sphere angles",
    )
    group_component_fig, group_component_axes = plot_spin_components(
        J.t,
        J.x,
        J.y,
        J.z,
        J.length,
        x_sem=J.x_sem,
        y_sem=J.y_sem,
        z_sem=J.z_sem,
        length_sem=J.length_sem,
        labels=labels,
        shade_index=2,
        linestyle=linestyle,
        axes=group_component_axes,
        phase_protocol=metadata.phase_protocol,
        title="Group-resolved J-sphere moments",
    )

# Keep time labels clear of the shared figure legends.
group_angle_fig.supxlabel("")
group_angle_axes[-1].set_xlabel(r"$\Gamma t$")
group_component_fig.supxlabel("")
for ax in group_component_axes[2:]:
    ax.set_xlabel(r"$\Gamma t$")

if save_plots:
    group_angle_fig.savefig(
        output_dir / "group_j_angles.png", dpi=200, bbox_inches="tight"
    )
    group_component_fig.savefig(
        output_dir / "group_j_moments.png", dpi=200, bbox_inches="tight"
    )

## Bright and dark modes

In [ ]:
mode_names = ("bright", "dark")
mode_axes = None
for method_name, moments, linestyle in (
    ("MCWF", mcwf_moments, "-"),
    ("MFE", mfe_moments, "--"),
):
    modes = moments.J_modes
    vectors = tuple(getattr(modes, name) for name in mode_names)
    mode_fig, mode_axes = plot_spin_components(
        modes.t,
        tuple(vector.x for vector in vectors),
        tuple(vector.y for vector in vectors),
        tuple(vector.z for vector in vectors),
        tuple(vector.length for vector in vectors),
        x_sem=tuple(vector.x_sem for vector in vectors),
        y_sem=tuple(vector.y_sem for vector in vectors),
        z_sem=tuple(vector.z_sem for vector in vectors),
        length_sem=tuple(vector.length_sem for vector in vectors),
        labels=tuple(f"{method_name} {name}" for name in mode_names),
        phase_protocol=moments.metadata.phase_protocol,
        colour_family_index=None,
        shade_index=2,
        linestyle=linestyle,
        axes=mode_axes,
        spin_symbol="s",
        title=f"MCWF and MFE bright/dark modes",
    )

# Per-panel time labels leave the shared figure legend unobstructed.
mode_fig.supxlabel("")
for ax in mode_axes[2:]:
    ax.set_xlabel(r"$\Gamma t$")

if save_plots:
    output_dir.mkdir(parents=True, exist_ok=True)
    mode_fig.savefig(output_dir / "collective_modes.png", dpi=200, bbox_inches="tight")

## Phase-2 dark-mode traces

In [ ]:
phase_2_traces = {}
dark_axes = None
for method_name, moments, linestyle in (
    ("MCWF", mcwf_moments, "-"),
    ("MFE", mfe_moments, "--"),
):
    dark = moments.J_modes.dark
    phase_t, phase_series = extract_family_phase_series(
        moments.J_modes.t,
        (dark.x, dark.y, dark.z, dark.length),
        moments.metadata.phase_protocol,
        family_phase_index=1,
    )
    if dark.x_sem is None:
        phase_sems = (None, None, None, None)
    else:
        _, phase_sems = extract_family_phase_series(
            moments.J_modes.t,
            (dark.x_sem, dark.y_sem, dark.z_sem, dark.length_sem),
            moments.metadata.phase_protocol,
            family_phase_index=1,
        )
    phase_2_traces[method_name] = (phase_t, phase_series[:3])
    dark_fig, dark_axes = plot_spin_components(
        phase_t,
        *phase_series,
        x_sem=phase_sems[0],
        y_sem=phase_sems[1],
        z_sem=phase_sems[2],
        length_sem=phase_sems[3],
        labels=(f"{method_name} dark",),
        phase_protocol=moments.metadata.phase_protocol,
        colour_family_index=1,
        shade_index=2,
        linestyle=linestyle,
        axes=dark_axes,
        spin_symbol="s",
        title=f"MCWF and MFE dark modes during phase 2",
    )

phase_2_t, phase_2_dark_components = phase_2_traces["MCWF"]
mfe_phase_2_t, mfe_phase_2_dark_components = phase_2_traces["MFE"]
for ax in dark_axes:
    ax.set_xlim(phase_2_t[0], phase_2_t[-1])
dark_fig.supxlabel("")
for ax in dark_axes[2:]:
    ax.set_xlabel(r"$\Gamma t$")

if save_plots:
    dark_fig.savefig(output_dir / "phase_2_dark_mode.png", dpi=200, bbox_inches="tight")

## Dark-mode harmonic analysis

In [ ]:
spectral_components = ("x", "y", "z")
dark_harmonic_analysis = {}
mfe_dark_harmonic_analysis = {}
for method_name, phase_t, components, analyses in (
    ("MCWF", phase_2_t, phase_2_dark_components, dark_harmonic_analysis),
    ("MFE", mfe_phase_2_t, mfe_phase_2_dark_components, mfe_dark_harmonic_analysis),
):
    for component, values in zip(spectral_components, components):
        analysis = compute_harmonic_analysis(
            phase_t,
            values,
            max_harmonic=5,
        )
        analyses[component] = analysis
        if analysis is None:
            print(f"Skipped {method_name} dark {component}: less than one resolved oscillation.")

In [ ]:
harmonic_fig = None
harmonic_axes = None
for method_index, (method_name, analyses, linestyle) in enumerate((
    ("MCWF", dark_harmonic_analysis, "-"),
    ("MFE", mfe_dark_harmonic_analysis, "--"),
)):
    fitted_components = tuple(
        component for component in spectral_components
        if analyses[component] is not None
    )
    if not fitted_components:
        continue
    fitted_signals = tuple(analyses[component].signals[0] for component in fitted_components)
    component_positions = [spectral_components.index(component) for component in fitted_components]
    harmonic_fig, harmonic_axes = plot_harmonic_sweep(
        component_positions,
        [signal.fundamental_frequency for signal in fitted_signals],
        [signal.total_harmonic_distortion for signal in fitted_signals],
        [signal.rms_amplitude for signal in fitted_signals],
        [signal.offset for signal in fitted_signals],
        labels=(method_name,),
        parameter_label="Dark-mode component",
        family_phase_index=1,
        colour_family_index=method_index,
        shade_index=2,
        marker="o",
        linestyle=linestyle,
        axes=harmonic_axes,
        title=f"MCWF and MFE dark-mode harmonics ({delta_label})",
    )

if harmonic_fig is None:
    print("No component has a resolved phase-2 harmonic fit.")
else:
    for ax in harmonic_axes:
        ax.set_xticks(
            np.arange(len(spectral_components)),
            [rf"$s_{{{component}}}$" for component in spectral_components],
        )
        ax.set_xlim(-0.5, len(spectral_components) - 0.5)
    if save_plots:
        harmonic_fig.savefig(output_dir / "dark_mode_harmonics.png", dpi=200, bbox_inches="tight")

## Analysis summary

In [ ]:
{
    "artifact": str(artifact_path),
    "Ni": metadata.Ni,
    "couplings": metadata.omega_groups,
    "ntraj": ntraj,
    "initial sectors": sectors,
    "delta/(N*Gamma)": delta_factor,
    "MFE solve succeeded": mfe_result.success,
    "MCWF/MFE saved grids match": np.array_equal(mcwf_moments.t, mfe_moments.t),
    "phase-2 samples": phase_2_t.size,
    "MCWF fitted dark components": tuple(
        component for component, analysis in dark_harmonic_analysis.items()
        if analysis is not None
    ),
    "MFE fitted dark components": tuple(
        component for component, analysis in mfe_dark_harmonic_analysis.items()
        if analysis is not None
    ),
}